# SteerMoE: forcing number formatting on Qwen3-30B-A3B

Replicates **"Steering MoE LLMs via Expert (De)Activation"** ([arXiv:2509.09660](https://arxiv.org/abs/2509.09660), [official code](https://github.com/adobe-research/SteerMoE)) on **Qwen3-30B-A3B** (48 MoE layers × 128 experts, top-8), one of the models evaluated in the paper, end to end in one engine:

1. **Detection** — per-token router logits are captured for contrastive pairs (answering with digits `1, 2, 3` vs. words `one, two, three`) with EasySteer's `router_logits` capture stream, each expert's top-k selection rate on the behavior tokens yields the risk difference `Δ = p_digits − p_words`, and the top word-linked experts are saved as a `deactivate` steering config (`steermoe_qwen3_words.json`).
2. **Steering** — compare greedy counting with and without deactivating those experts, then count any deactivated-expert selections in the steered router outputs.

Qwen3-30B-A3B is a hybrid thinking model; prompts render with `enable_thinking=False`, as in the official SteerMoE code.

**Execution note:** Saved outputs are from earlier runs and are retained for reference. The updated code has not been rerun here.


In [1]:
import json
import os

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("VLLM_LOGGING_LEVEL", "WARNING")  # quiet engine boot logs

import numpy as np
import easysteer.hidden_states as hs
from vllm import LLM, SamplingParams
from vllm.capture import SelectSpec
from vllm.steer_vectors import ApplySpec, SteeringSpec, VectorSpec
from transformers import AutoConfig

MODEL = os.environ.get("EASYSTEER_MODEL", "Qwen/Qwen3-30B-A3B")
hf_cfg = AutoConfig.from_pretrained(MODEL).to_dict()
N_EXPERTS = hf_cfg["num_experts"]      # 128
TOP_K = hf_cfg["num_experts_per_tok"]  # 8

llm = LLM(
    model=MODEL,
    enable_steer_vector=True,
    steer_algorithms=["moe_router"],
    # This notebook only uses deactivate, which supports in-graph steering.
    steer_graph_mode="in_graph",
    gpu_memory_utilization=0.92,
    max_model_len=4096,
    max_num_seqs=4,
)
tok = llm.get_tokenizer()

/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/pydantic/dataclasses.py:313: UserWarning: `config` is set via both the `dataclass` decorator and `__pydantic_config__` for dataclass SteerVectorConfig. The `config` specification from `dataclass` decorator will take priority.
  return create_dataclass if _cls is None else create_dataclass(_cls)


WARNING 08-05 20:10:51 [arg_utils.py:2652] This model does not officially support disabling chunked prefill. Disabling this manually may cause the engine to crash or produce incorrect outputs.


WARNING 08-05 20:10:51 [vllm.py:1199] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none


WARNING 08-05 20:10:51 [vllm.py:1249] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


...

WARNING 08-05 20:16:57 [fused_moe.py:1107] Using default MoE config. Performance might be sub-optimal! Config file not found at /data/zju-48b/xhl/anaconda3/envs/easysteer-vllm026/lib/python3.12/site-packages/vllm/model_executor/layers/fused_moe/configs/E=128,N=768,device_name=NVIDIA_RTX_PRO_5000_72GB_Blackwell.json


(EngineCore pid=3834982) 

WARNING 08-05 20:17:18 [vllm.py:1199] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none


(EngineCore pid=3834982) 

WARNING 08-05 20:17:18 [vllm.py:1249] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.


## Expert detection

### The contrastive pairs

Each side renders a full chat turn **including the assistant response**, so
a single prefill routes every response token through the MoE layers. The
`target` string marks the tokens whose routings we compare. The official
demo uses a single pair; a few pairs sharpen the risk difference
considerably.

In [2]:
PAIRS = [
    ("Count to ten",
     "1, 2, 3, 4, 5, 6, 7, 8, 9, 10",
     "one, two, three, four, five, six, seven, eight, nine, ten"),
    ("How many days are in a week, and how many months in a year?",
     "There are 7 days in a week and 12 months in a year.",
     "There are seven days in a week and twelve months in a year."),
    ("What is five plus three?",
     "5 + 3 = 8",
     "five plus three equals eight"),
]

### Capture router logits

`hs.capture(..., stream="router_logits")` returns labeled rows grouped by
sample and true model layer ID. Select the answer's token positions at
capture time; the helper manages the capture lifecycle while the engine
handles graph execution and prefix-cache reads.

In [3]:
def find_sub_list(sub, seq):
    n = len(sub)
    return [(i, i + n - 1) for i in range(len(seq) - n + 1)
            if seq[i:i + n] == sub]


def topk_membership(rows):
    """(tokens, n_experts) logits -> bool top-k membership mask."""
    order = np.argsort(rows, axis=-1)[:, -TOP_K:]
    mask = np.zeros(rows.shape, dtype=bool)
    np.put_along_axis(mask, order, True, axis=-1)
    return mask


counts = {"digits": None, "words": None}
totals = {"digits": 0, "words": 0}
layer_ids = None
for user, digits_ans, words_ans in PAIRS:
    for key, answer in (("digits", digits_ans), ("words", words_ans)):
        msgs = [{"role": "user", "content": user},
                {"role": "assistant", "content": answer}]
        prompt_ids = tok.apply_chat_template(
            msgs, tokenize=True, return_dict=False, add_generation_prompt=False,
            enable_thinking=False,
        )
        target_ids = tok(answer, add_special_tokens=False).input_ids
        s, e = find_sub_list(target_ids, prompt_ids)[-1]
        result = hs.capture(
            llm, [{"prompt_token_ids": prompt_ids}],
            stream="router_logits",
            select=SelectSpec(prompt_window=(s, e + 1)),
            steering=False,
        )
        if layer_ids is None:
            layer_ids = result.layer_ids
        if result.layer_ids != layer_ids:
            raise RuntimeError("captured MoE layer IDs changed between samples")
        if result.sample_positions(0) != list(range(s, e + 1)):
            raise RuntimeError("capture did not return every target token position")
        logits = result.sample(0)

        # Statistics use array rows; layer_ids maps them to model layer IDs.
        sel = np.stack([topk_membership(logits[lid].float().numpy())
                        for lid in layer_ids])
        cnt = sel.sum(axis=1)  # (layer row, expert)
        counts[key] = cnt if counts[key] is None else counts[key] + cnt
        totals[key] += e - s + 1

print(f"detection tokens: digits={totals['digits']} "
      f"words={totals['words']}")

detection tokens: digits=53 words=38


### Risk difference

`Δ(layer, expert) = p_digits − p_words`: experts with large positive Δ are
selected for digit tokens but not word tokens.

In [4]:
risk_diff = counts["digits"] / totals["digits"] \
    - counts["words"] / totals["words"]

flat = np.argsort(np.abs(risk_diff), axis=None)[::-1]
print("top behavior-linked experts (layer, expert, Δ):")
for idx in flat[:10]:
    row, expert = divmod(int(idx), N_EXPERTS)
    layer = layer_ids[row]
    print(f"  L{layer:02d} E{expert:02d}  "
          f"Δ={risk_diff[row, expert]:+.2f}")

top behavior-linked experts (layer, expert, Δ):
  L40 E115  Δ=-0.51
  L43 E75  Δ=-0.51
  L05 E22  Δ=-0.49
  L45 E20  Δ=-0.48
  L44 E64  Δ=-0.45
  L45 E41  Δ=+0.45
  L00 E85  Δ=+0.45
  L03 E54  Δ=-0.44
  L42 E30  Δ=-0.43
  L00 E34  Δ=-0.42


### Save the steering config

Deactivating the **word-linked** experts (negative Δ) steers away from
spelled-out numbers. Keep the experiment's 200 deactivated experts
(~3% of 128×48); the saved outputs below record the earlier run's shift
toward digits. The paper tunes this count per model and task (Table A.2).
The config uses true model layer IDs, even when captured layers are
non-contiguous.

In [5]:
N_DEACT = 200

deact = {}
taken = 0
for idx in flat:
    row, expert = divmod(int(idx), N_EXPERTS)
    layer = layer_ids[row]
    # word-linked experts have negative delta (digits_rate - words_rate)
    if risk_diff[row, expert] >= 0:
        continue
    deact.setdefault(layer, []).append(expert)
    taken += 1
    if taken == N_DEACT:
        break

with open("steermoe_qwen3_words.json", "w") as f:
    json.dump({"layer_configs": {
        str(layer): {"mode": "deactivate", "expert_ids": ids}
        for layer, ids in deact.items()
    }}, f, indent=2)
print(f"saved steermoe_qwen3_words.json: {taken} experts "
      f"across {len(deact)} layers")

saved steermoe_qwen3_words.json: 200 experts across 46 layers


## Steering

In [6]:
with open("steermoe_qwen3_words.json") as f:
    layer_configs = json.load(f)["layer_configs"]

steering = SteeringSpec(vectors=[
    VectorSpec(
        source="steermoe_qwen3_words.json",
        algorithm="moe_router",  # per-layer mode/expert_ids come from the JSON
        layers=sorted(int(la) for la in layer_configs),
        apply=ApplySpec(prompt="all", generation="all"),
    ),
])


def render(prompt):
    return tok.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=True, return_dict=False,
        add_generation_prompt=True, enable_thinking=False,
    )


def gen(prompt, spec=False):
    out = llm.generate({"prompt_token_ids": render(prompt)},
                       sampling_params=SamplingParams(temperature=0.0,
                                                      max_tokens=64),
                       steering=spec,
                       use_tqdm=False)
    return out[0].outputs[0].text.strip().replace("\n", " ")


def digit_share(text):
    digits = sum(c.isdigit() for c in text)
    letters = sum(c.isalpha() for c in text)
    return digits / max(1, digits + letters)

In [7]:
PROMPTS = [
    "Count to fifteen.",
    "Count from one to twelve.",
]

for prompt in PROMPTS:
    base = gen(prompt)
    steered = gen(prompt, steering)
    print(f"[{prompt}]")
    print(f"  baseline (digit-share {digit_share(base):.2f}): {base}")
    print(f"  steered  (digit-share {digit_share(steered):.2f}): {steered}")

[Count to fifteen.]
  baseline (digit-share 0.17): Sure! Here's the count from one to fifteen:  1. One   2. Two   3. Three   4. Four   5. Five   6. Six   7. Seven   8. Eight   9. Nine   10. Ten   11. Eleven   12. Twelve   13
  steered  (digit-share 1.00): 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15.


[Count from one to twelve.]
  baseline (digit-share 0.00): One, two, three, four, five, six, seven, eight, nine, ten, eleven, twelve.
  steered  (digit-share 1.00): 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12.


### Mechanism check

Capture the post-steering router logits for one steered prompt and
count deactivated experts in each token's top-8. A count of zero confirms
that no deactivated expert was selected in these captured rows.

In [8]:
deact = {int(la): c["expert_ids"] for la, c in layer_configs.items()}

result = hs.capture(
    llm, [{"prompt_token_ids": render("Count to fifteen.")}],
    max_tokens=32,
    stream="router_logits",
    select=SelectSpec(prompt="all", generation="all"),
    steering=steering,
)
logits = {lid: tensor.float().numpy()
          for lid, tensor in result.sample(0).items()}

leaks = 0
for layer, expert_ids in deact.items():
    top = np.argsort(logits[layer], axis=-1)[:, -TOP_K:]
    leaks += int(np.isin(top, expert_ids).sum())
print(f"deactivated-expert selections post-steering: {leaks} "
      f"(0 = steering is airtight)")

deactivated-expert selections post-steering: 0 (0 = steering is airtight)
